# T05 — Synthetische Testdaten

T04 hat die Pipeline mit 7 von Hand geschriebenen Fragen evaluiert. Diese decken die bekannten Fehlerfälle ab, aber nicht die Breite des Korpus. Hunderte Fragen von Hand zu schreiben ist aufwendig, deshalb lassen wir ein LLM Paare aus `(question, reference answer)` aus den indexierten Chunks erzeugen.

1. **Handgeschriebene Beispiele**: wenige, gezielt auf bekannte Fehlerfälle, gegen die Dokumente geprüft
2. **Synthetische Beispiele**: Ein LLM schreibt pro ausgewähltem Chunk eine Frage und eine Antwort
3. **Export**: beide Sets im RAGAS-Format (`user_input`, `reference`)

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

Benötigt `OPENAI_API_KEY` (lokal: in `.env.local` im Hauptverzeichnis des Repos). Zuerst T01 ausführen: Dort entsteht der Vector Store, den die späteren Notebooks wiederverwenden.

In [ ]:
import json
import random
import sys

from loguru import logger

from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from ki_kompass_rag.baseline_rag import DB_DIR, build_llm, load_or_build_vector_store
from ki_kompass_rag.evaluation import EVALUATION_QUERIES

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, _ = await load_or_build_vector_store()
llm = build_llm("openai")

## 1. Handgeschriebene Beispiele

Handgeschriebene Beispiele bilden die fachliche Absicht ab: Sie prüfen, ob das System noch die Lara-Palette erfindet oder eine Lieferantenangabe als verifiziert behandelt. Jedes braucht die genaue Frage, die getestete Kategorie und eine kurze richtige Antwort, die jemand gegen die Dokumente geprüft hat.

Die Fragen zu Fehlerfällen stammen aus `EVALUATION_QUERIES` (dem Testset aus T04); hier kommen einige direkte Faktenfragen dazu.

In [ ]:
manual_samples = [
    {
        "user_input": q["query"],
        "reference": q["ground_truth_answer"],
        "category": q["category"],
    }
    for q in EVALUATION_QUERIES
] + [
    {
        "user_input": "What is the total climate change GWP of the IPG F4090-05 machine roll tape per square meter?",
        "reference": "2.03E-01 kg CO2 eq. per m2: 1.29E-01 upstream, 6.69E-02 core and 7.50E-03 downstream.",
        "category": "direct_fact",
        "sources": ["EPD_tape_IPG_hotmelt.pdf"],
    },
    {
        "user_input": "What is the weight and static load capacity of the StabilPlastik EP 08 pallet?",
        "reference": "It weighs 25 kg and has a static load capacity of 10,000 kg.",
        "category": "direct_fact",
        "sources": ["EPD_pallet_stabilplastik_ep08.pdf"],
    },
    {
        "user_input": "How do the EU climate reporting guidelines define transition risks?",
        "reference": "Risks to the company that arise from the transition to a low-carbon and climate-resilient economy, such as policy, legal, technology, market and reputational risks.",
        "category": "direct_fact",
        "sources": ["REF_eu_climate_reporting_guidelines.pdf"],
    },
    {
        "user_input": "What is the Logypal 1 made of according to its 2021 datasheet?",
        "reference": "100% post-consumer recycled plastic, mainly from end-of-life agricultural packaging (silage film) and industrial packaging waste.",
        "category": "direct_fact",
        "sources": ["ART_relicyc_logypal1_datasheet_2021.pdf"],
    },
]
print(f"{len(manual_samples)} hand-written samples")

## 2. Synthetische Beispiele

Chunks aus dem Vector Store auswählen, sehr kurze (Titel, Fusszeilen) überspringen, und das LLM pro Chunk um eine Frage und eine Antwort bitten, die sich nur auf diesen Chunk stützen.

`GENERATOR_PROMPT` in der zweiten Zelle übersetzt:

> Schreibe eine Frage und eine Antwort, die sich strikt auf den Kontext unten stützen.
>
> - Frage, was eine Einkäuferin oder ein Nachhaltigkeitsauditor fragen würde.
> - Die Antwort muss sachlich und knapp sein und nur aus dem Kontext stammen.
> - Nenne konkrete Produkte, IDs oder Daten aus dem Kontext.
>
> Antworte genau in diesem Format:
> QUESTION: <Frage>
> ANSWER: <Antwort>
>
> Kontext:
> {context}

In [ ]:
SAMPLE_SIZE = 10

random.seed(42)
records = await vector_store.get_chunks_by_filter()
candidates = [r for r in records if len(r.content) > 300]
sampled = random.sample(candidates, min(SAMPLE_SIZE, len(candidates)))
print(f"Sampled {len(sampled)} of {len(candidates)} chunks")

In [ ]:
GENERATOR_PROMPT = """Write one question and answer based strictly on the context below.

- Ask what a procurement officer or sustainability auditor would ask.
- The answer must be factual, concise and derived only from the context.
- Mention specific products, IDs or dates from the context.

Reply exactly in this format:
QUESTION: <question>
ANSWER: <answer>

Context:
{context}"""

synthetic_samples = []
for record in sampled:
    prompt = GENERATOR_PROMPT.format(context=record.content)
    response = await llm.generate(
        [
            LLMMessage(
                role=Roles.USER, content=[MessageContent(type="text", text=prompt)]
            )
        ]
    )
    text = response.content[0].text or ""
    if "ANSWER:" not in text:
        print(f"Skipped unparsable response for {record.metadata['source_file']}")
        continue
    question, answer = text.split("ANSWER:", 1)
    synthetic_samples.append(
        {
            "user_input": question.replace("QUESTION:", "").strip(),
            "reference": answer.strip(),
            "category": "synthetic",
            "sources": [record.metadata["source_file"]],
        }
    )

for sample in synthetic_samples[:3]:
    print(
        f"[{sample['sources'][0]}]\nQ: {sample['user_input']}\nA: {sample['reference']}\n"
    )

Synthetische Fragen übernehmen die Formulierung des Chunks; das macht das Retrieval leichter als bei echten Fragen, und das LLM kann den Chunk falsch verstehen. Eine Stichprobe prüfen, bevor man sich darauf verlässt.

## 3. Export

In [ ]:
all_samples = manual_samples + synthetic_samples
output_path = DB_DIR / "evaluation_dataset.json"
output_path.write_text(json.dumps(all_samples, indent=2, ensure_ascii=False))
print(f"Saved {len(all_samples)} samples to {output_path}")

Um damit zu evaluieren, in T04 `user_input` und `reference` anstelle von `query` und `ground_truth_answer` verwenden.